# dynalyst

One agent for bifurcation analysis, simulation, steady states, parameter estimation,
model classification, SBML annotations and custom plots.

How a query runs: the **orchestrator** writes a plan and delegates each step -> a **team
supervisor** -> a **sub-agent** calls its analysis **tools** and reads their results
(errors included) -> the orchestrator answers from those results -> **final-answer check**
(every number, model ID and title must come from a tool result).

Run the first cell, **choose the model** from the list, and paste its API key if asked; then use the chat box or `ask(...)`.
**Clear chat** also clears the session's memory of earlier questions.

In [ ]:
import os, sys
from pathlib import Path

# Make the project importable (not needed after `pip install -e .`).
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynalyst import config
from dynalyst.env_check import check_env
from dynalyst.model_picker import pick_model
check_env()      # names any package requirements.txt asks for that is missing
print("AUTO-07p:", config.AUTO_DIR)

# Choose the model from the list, then give its API key (asked only if it needs one; a key
# already set in the environment is used when the box is left blank). Nothing is saved to disk.
pick_model();


## Upload models and articles
`.xml` SBML files are saved to `data/sbml_models/`; `.txt`/`.ode` XPP files are converted to SBML there; `.pdf` articles go to `data/papers/`, where the agent can read them and find their BioModels entries.

In [2]:
import json, traceback
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
from dynalyst import config
from dynalyst.analysis import xpp_to_sbml

MODELS = config.LOCAL_SBML_MODELS_DIR
MODELS.mkdir(parents=True, exist_ok=True)
uploader = widgets.FileUpload(accept=".xml,.txt,.ode,.pdf", multiple=True,
                             description="Upload model(s) / article(s)")
log = widgets.Output()  # prints from a widget callback are hidden unless captured here


def _save(name, content):
    if Path(name).suffix.lower() == ".pdf" or content.startswith(b"%PDF"):
        config.PAPERS_DIR.mkdir(parents=True, exist_ok=True)
        (config.PAPERS_DIR / name).write_bytes(content)
        return f"Article saved: {config.PAPERS_DIR / name} (ask about it, or for its BioModels entry)"
    if Path(name).suffix.lower() == ".xml" or content.lstrip().startswith((b"<?xml", b"<sbml")):
        (MODELS / name).write_bytes(content)
        return f"SBML saved: {MODELS / name}"
    result = json.loads(xpp_to_sbml.xpp_text_to_sbml(
        text=content.decode("utf-8", errors="replace"), model_name=Path(name).stem))
    if "error" in result:
        return f"ERROR converting {name}: {result['error']}"
    return f"XPP converted: {result['path']}"


def _process_uploaded_models(change):
    values = change["new"]
    with log:
        for file_info in (values.values() if isinstance(values, dict) else values) or []:
            name = file_info["name"]
            try:
                print(_save(name, bytes(file_info["content"])))
            except Exception:
                print(f"ERROR saving {name}:")
                traceback.print_exc()


uploader.observe(_process_uploaded_models, names="value")
display(uploader, log)
print("Models folder:", MODELS)
print("Already there:", sorted(f.name for f in MODELS.glob("*.xml")))

# If the upload button does nothing in your editor, copy the file into the
# folder above instead, or convert an XPP file directly:
# print(_save("my_model.ode", Path("/path/to/my_model.ode").read_bytes()))


FileUpload(value=(), accept='.xml,.txt,.ode,.pdf', description='Upload model(s) / article(s)', multiple=True)

Output()

Models folder: /E-drive/ABHILASHA/CRNT_agent_new/CRNT_agent/jaxkm_bifurcation_agent/data/sbml_models
Already there: ['BIOMD0000000010.xml', 'Bachmann_MSB2011.xml', 'Becker_Science2010.xml', 'Beer_MolBioSystems2014.xml', 'Bertozzi2020.xml', 'Boehm_JProteomeRes2014.xml', 'Borghans_BiophysChem1997.xml', 'Bruno_JExpBio2016.xml', 'Elowitz_Nature2000.xml', 'Fiedler_BMC2016.xml', 'Full_model_for_SN.xml', 'Full_model_for_SN_plus_TB.xml', 'Garde2020.xml', 'Markevich2004.xml', 'Sir_paper_fixed.xml', 'The_parameter_gives_PF_bifurcation.xml', 'brannmark.xml', 'p_dynamics.xml', 'pitchfork.xml', 'saddlenode.xml', 'srirammarkfullALL.xml', 'transcritical.xml', 'user_model.xml']


## Chat

In [3]:
from dynalyst.chat_ui import launch_chat_ui
launch_chat_ui();   # ';': display the chat box once


## Single queries
Each call shows the plan, every delegation and tool call, the answer, a **What next?** list of follow-up questions, and the plots/tables it produced. In the chat box above, each follow-up is a button.

In [4]:
from dynalyst.notebook import ask

**Model classifier**

In [5]:
ask("What is the dimension of Markevich2004 and which bifurcations are possible in it?")

PLAN: [model_classifier] Determine the dimension of the Markevich2004 model, including state variables, conservation laws, independent dimension (1D / 2D / 3D+), and the dynamics that it allows.; [bifurcation] Analyze the bifurcations possible in the Markevich2004 model, including how the steady states change as one parameter varies, and provide bifurcation diagrams with folds, bistability, branch points, and Hopf points.
  DELEGATE -> model_classifier: Determine the dimension of the Markevich2004 model, including state variables, conservation laws, independent dimension (1D / 2D / 3D+), and the dynamics that it allows.
    SELECT classify_model({"model_name_or_path": "Markevich2004"})  guard: ok
    REVIEW accept
  DELEGATE -> bifurcation: Analyze the bifurcations possible in the Markevich2004 model, including how the steady states change as one parameter varies, and provide bifurcation diagrams with folds, bistability, branch points, a
    SELECT lhs_bifurcation_survey({"model_name_o

Before running AUTO-07p on **Markevich2004**: no continuation parameter was named, so here are the ranked candidates (eigenvalue screen along the settled equilibrium (analysis/parameter_screen.py); ranked by closeness of the crossing to the model's own value):

1. MAPKK over [25, 86.07], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at MAPKK ~ 57.38 (model value 50)
2. MKP3 over [43.57, 150], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at MKP3 ~ 87.14 (model value 100)
3. k4cat over [0.02326, 0.09], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at k4cat ~ 0.04652 (model value 0.06)
4. k1cat over [0.005, 0.01935], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at k1cat ~ 0.0129 (model value 0.01)
5. k3cat over [0.03122, 0.126], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at k3cat ~ 0.06243 (model value 0.084)

Reply **yes** to run candidate 1, a **number** or a **parameter name** to run another, or **no** to keep this report only (no AUTO run).

"Before running AUTO-07p on **Markevich2004**: no continuation parameter was named, so here are the ranked candidates (eigenvalue screen along the settled equilibrium (analysis/parameter_screen.py); ranked by closeness of the crossing to the model's own value):\n\n1. MAPKK over [25, 86.07], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at MAPKK ~ 57.38 (model value 50)\n2. MKP3 over [43.57, 150], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at MKP3 ~ 87.14 (model value 100)\n3. k4cat over [0.02326, 0.09], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at k4cat ~ 0.04652 (model value 0.06)\n4. k1cat over [0.005, 0.01935], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at k1cat ~ 0.0129 (model value 0.01)\n5. k3cat over [0.03122, 0.126], plotting Mp -- expected LP (fold); fold (equilibrium disappears) at k3cat ~ 0.06243 (model value 0.084)\n\nReply **yes** to run candidate 1, a **number** or a **parameter name** t

**Steady states**

In [ ]:
ask("Find the steady states of Markevich2004 and their stability")

**Simulation**

In [ ]:
ask("Simulate the Garde2020 model for 100 time units and summarize the dynamics")

**Bifurcation**

In [ ]:
ask("Run a bifurcation analysis of Markevich2004.xml in k2cat and tell me where the folds are")

In [ ]:
ask("LHS sample 20 points of Markevich2004.xml continuing in k2cat, varying only Km5 within a factor of 2, and split them into categories")

In [ ]:
ask("Bifurcation diagram of pdp_mm_bistable.txt in ATOT")

In [ ]:
ask("plot the generic model of saddle node")

**Model reduction (QSSA)**: the agent checks timescales and only applies a QSSA when the gap justifies it; the result is validated against the full model.

In [ ]:
ask("Does Full_model_for_SN have a timescale separation? Reduce it with QSSA and tell me how accurate the reduced model is")

**Annotations**

In [ ]:
ask("What annotations does the Markevich2004 model carry?")

### Bistability from reaction network theory (crnt4sbml)
The model is converted to a crnt4sbml-compatible SBML automatically; CRNT only: it reports rate constants with several steady states; checking bistability with AUTO-07p, an LHS scan or the network drawing are offered afterwards.

In [ ]:
ask("Can Full_model_for_SN be bistable? Use reaction network theory")

In [ ]:
ask("Which bifurcations does Garde2020.xml show across its parameter space? Run an LHS survey")

### BioModels and uploaded articles

In [ ]:
ask("Search BioModels for curated models of bistability in MAPK signalling")

In [ ]:
ask("Which BioModels entries come from the article garde.pdf? Download the first one and classify it")

In [ ]:
ask("What does the article elowitz.pdf say about how the repressilator was built?")

**Custom plot**

In [ ]:
ask("Plot A and APP of pitchfork_model.xml against time up to t=2000, comparing KTOT=100 with KTOT=200")

**Parameter estimation**

The next cell writes a synthetic time course from `pitchfork_model.xml` with k4 doubled and
k8 halved, so there is something to fit. Replace it with your own CSV: one row per time point,
a `time` column, one column per measured species.

In [ ]:
from dynalyst import config
import numpy as np, pandas as pd, jax.numpy as jnp
from jaxkineticmodel.load_sbml.sbml_model import SBMLModel
from dynalyst.analysis.jaxkm_compat import jaxkm_ready_sbml

m = SBMLModel(str(jaxkm_ready_sbml(config.MODELS_DIR / "pitchfork_model.xml")))
true = dict(m.parameters); true["k4"] *= 2; true["k8"] *= 0.5
ts = np.linspace(0, 400, 41)
ys = np.asarray(m.get_kinetic_model()(jnp.asarray(ts), m.y0, true))
data = config.DATA_DIR / "timecourses" / "pitchfork_synthetic.csv"
data.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame({"time": ts, **{s: ys[:, i] for i, s in enumerate(m.species_names)}}).to_csv(data, index=False)
print("wrote", data, "| true k4, k8 =", true["k4"], true["k8"])

In [ ]:
ask("Fit k4 and k8 of pitchfork_model.xml to the time course in pitchfork_synthetic.csv using 10 Latin-hypercube start points, and tell me whether they are identifiable")

## Inspect a run
The full record (every plan version, every tool call with its result or error, where a code exception stepped in, the final check) is also saved as JSON in `outputs/runs/`; `python scripts/inspect_run.py` prints the latest one.

In [ ]:
from dynalyst.agents.unified_agent import run

rec = run("Find the steady states of Markevich2004", verbose=False)
for v in rec.plan_versions:
    print(f"plan v{v['version']}" + (f" ({v['reason']})" if v["reason"] else "") + ":", v["steps"])
for s in rec.steps:
    print(f"[{s.get('sub_agent') or 'orchestrator'}] {s['tool']} -> {s['status']}"
          + (f": {s['error'][:150]}" if s.get("error") else ""))
print("
exceptions:", rec.interventions)
print("final check:", rec.final_check, "| model calls:", rec.model_calls, "| record:", rec.record_path)